In [ ]:
import polars as pl
import pyranges as pr

In [ ]:
anno = pl.scan_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass1e6_genes_10kb_allvars_annotated_250923.parquet")

anno_sub = anno.select(['chrom', 'pos', 'ref', 'alt', 'id', 'region']).head().collect()
anno_sub

In [ ]:
# Path to your GENCODE GTF file
gtf_path = "/s/project/deeprvat/ukb_gym/gencode/gencode.v39.annotation.gtf.gz"

# Read GTF and filter for transcripts
# Using as_df=True is efficient as we'll convert to polars right away
gencode_pr = pr.read_gtf(gtf_path, as_df=True)
gencode_pl = (
    pl.from_pandas(gencode_pr)
    .filter(
        (pl.col("gene_type") == "protein_coding")
    )
)

gencode_pl

In [ ]:
gencode_genes = gencode_pl.filter(pl.col("Feature") == "gene").with_columns(
    region = pl.col("gene_id").str.split('.').list.first(),
)
gencode_genes

In [ ]:
tss_df = gencode_genes.with_columns(
    gene_length = pl.col("End") - pl.col("Start") + 1,
    TSS = pl.when(pl.col("Strand") == "+")
            .then(pl.col("Start"))
            .otherwise(pl.col("End"))
).select([
    "TSS",
    "Strand",
    "gene_length",
    "gene_name",
    "region",
    # "gene_id",
    # "Chromosome",
])
tss_df

In [ ]:
anno_tss = (
    anno.select(['chrom', 'pos', 'ref', 'alt', 'id', 'region'])
    .join(
        tss_df.lazy(), 
        on="region", 
        how="left"
    )
    .with_columns(
        dist_to_tss = pl.when(pl.col("Strand") == "+")
            .then(pl.col("pos") - pl.col("TSS"))
            .otherwise(pl.col("TSS") - pl.col("pos"))
    )
).collect(engine='streaming')

anno_tss

In [ ]:
anno_all = anno.join(anno_tss.lazy(), on=['chrom', 'pos', 'ref', 'alt', 'id', 'region'], how='left').sink_parquet("/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass1e6_genes_10kb_allvars_annotated_250925.parquet", engine='streaming')

## Debug

In [ ]:
anno_tss = anno_sub.join(tss_df, on="region", how="left")
anno_tss

In [ ]:
anno_tss.with_columns(
    dist_to_tss = pl.when(pl.col("Strand") == "+")
                    .then(pl.col("pos") - pl.col("TSS"))
                    .otherwise(pl.col("TSS") - pl.col("pos"))
)